# Workshop 2: teaching the engine to finish, and to be punished

The first workshop stopped at PPO and mate-in-two boards (2026-09-16). This one
covers what came after, up to the blunder audit of 2026-09-19: starting next to a
human mate, borrowing a demonstrator's move, playing against frozen copies of
yourself, generated bare-king endgames with a start-state curriculum, learning
twice from your own wins, lambda 1, and finally measuring whether the opponent
punishes gifts. Same format as before:

1. **Plain words** — the idea and a chess picture.
2. **The real code** — the lines from `src/` that implement it.
3. **Your turn** — fill in the `...`, run the cell, then run the `check_*` cell.

Rules: no peeking at `learning/workshop2_solutions.md` before a check fails
twice. Reflections go in the last cell and from there into `learning/records/`.

Run the setup cell first.

In [ ]:
import json, math, random
from collections import deque
from pathlib import Path
import torch
import matplotlib.pyplot as plt

torch.manual_seed(0); random.seed(0)
RESULTS = []

def check(name, ok, detail=""):
    RESULTS.append((name, bool(ok)))
    print(("✅ " if ok else "❌ ") + name + (f"  ({detail})" if detail else ""))

def close(a, b, tol=1e-3):
    a, b = torch.as_tensor(a, dtype=torch.float64), torch.as_tensor(b, dtype=torch.float64)
    return a.shape == b.shape and bool(torch.allclose(a, b, atol=tol, rtol=0))

def load_run(path):
    d = json.loads(Path(path).read_text())
    return d["logs"], d["config"]

def series(logs, key):
    xs = [e["episode"] for e in logs if e.get(key) is not None]
    ys = [e[key] for e in logs if e.get(key) is not None]
    return xs, ys

def chain(*paths):
    """Concatenate run logs, shifting update numbers so runs continue each other."""
    out, off = [], 0
    for p in paths:
        lg, cfg = load_run(p)
        out += [{**e, "episode": e["episode"] + off} for e in lg]
        off += cfg["max_updates"]
    return out

print("setup ok")

## 1. Start next to a human mate, and widen only when you deserve it

**Plain words.** Mate-in-one boards teach the last move. Real games need the
last ten. Florensa et al. (2017) start the agent next to the goal and move the
start backwards as the success rate allows. Here the goal is a human checkmate
from Lichess: a *finishing board* rewinds the game `k` plies before the mate,
with the winner to move, and plays on under a ply cap. `k` is drawn uniformly
from the even numbers up to `current_depth`, which rises by 2 whenever the last
`window` episodes succeeded at least `advance_rate` of the time. It never falls.

**Real code** — `FinishCurriculum.report` (`src/envs/start_positions.py`):

```python
def report(self, depth, success):
    self._recent.append(bool(success))
    if len(self._recent) > self.window:
        self._recent.pop(0)
    self._since_rise += 1
    if self._since_rise >= self.window and self.current_depth < self.depth_max:
        rate = sum(self._recent) / len(self._recent)
        if rate >= self.advance_rate:
            self.current_depth = min(self.current_depth + 2, self.depth_max)
            self._since_rise = 0
```

In [ ]:
# TASK: depth_curriculum
def next_depth(current, recent, since_rise, window, advance_rate, depth_max):
    """Return the new current_depth after one more report has been appended to `recent`
    (recent already trimmed to the last `window` outcomes; since_rise already incremented)."""
    if since_rise >= window and current < depth_max:
        rate = ...
        if rate >= advance_rate:
            return ...
    return current

# By hand: depth 2, window 100, advance_rate 0.6, depth_max 40.
# (a) 100 episodes reported since the last rise, 61 successes -> new depth?
# (b) same but 59 successes -> new depth?
MY_DEPTH_61 = ...
MY_DEPTH_59 = ...

In [ ]:
from src.envs.start_positions import FinishCurriculum, FinishRecord
_rec = FinishRecord("g", "6k1/5ppp/8/8/8/8/5PPP/R5K1 w - - 0 1", ["a1a8"], 1)
def _engine(successes):
    c = FinishCurriculum([_rec], depth_start=2, depth_max=40, advance_rate=0.6, window=100)
    for i in range(100):
        c.report(2, i < successes)
    return c.current_depth
check("next_depth advances at 61/100", next_depth(2, [True]*61 + [False]*39, 100, 100, 0.6, 40) == _engine(61) == 4)
check("next_depth holds at 59/100", next_depth(2, [True]*59 + [False]*41, 100, 100, 0.6, 40) == _engine(59) == 2)
check("hand answers", (MY_DEPTH_61, MY_DEPTH_59) == (_engine(61), _engine(59)), f"engine: {_engine(61)}, {_engine(59)}")

### 1b. The rule that never fired

The held-out finishing evaluation asks the network to convert from 2, 10 and 20
plies before the mate. Below is `finish_depth`, the training curriculum's
current depth, across LONG2 and LONG3 (1,200 updates). Read the plot, then answer.

In [ ]:
_fin = chain("experiments/endgame-technique/LONG2/run.json", "experiments/endgame-technique/LONG3/run.json")
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
xs, ys = series(_fin, "finish_depth"); axes[0].plot(xs, ys); axes[0].set_title("training curriculum depth (finish_depth)")
for k in ("finish_rate_d2", "finish_rate_d10", "finish_rate_d20"):
    xs, ys = series(_fin, k); axes[1].plot(xs, ys, marker="o", label=k)
axes[1].set_title("held-out finishing rate"); axes[1].legend()
for ax in axes: ax.set_xlabel("update")
plt.show()

In [ ]:
# TASK: read_depth
# 1. What is the deepest start (in plies before the mate) a training finishing board ever used?
DEEPEST_TRAINED = ...
# 2. Which of the evaluated depths did training therefore never visit? A list from [2, 10, 20].
NEVER_TRAINED = [...]
# 3. In one sentence: the curriculum counted every win, including wins made by the demonstrator's move.
#    Why does that make the rule *less* likely to fire, not more?  (Hint: what happens to the human line
#    once the defender leaves it, and who is the defender?)
WHY_STUCK = """..."""

In [ ]:
_deepest = max(series(_fin, "finish_depth")[1])
check("deepest trained depth", DEEPEST_TRAINED == _deepest, f"engine: {_deepest}")
check("never-trained depths", sorted(NEVER_TRAINED) == [d for d in (2, 10, 20) if d > _deepest])
print("your sentence:", WHY_STUCK)
print("expected direction: the defender is our own network and leaves the human line on ply 1-2, so the line stops"
      " helping; the 0.6 bar must then be met by the bare network at every depth up to current, which it never was.")

## 2. Borrow the demonstrator's move, but pay for it honestly

**Plain words.** Random exploration noise flattened the policy in both arms it was
tried on. Instead, when a *demonstration* exists for the position — the puzzle's
key move, the human's next move while the game still follows the record, or a
rules-engine mate in one — the behaviour policy plays it with probability
`epsilon` and the network's own draw otherwise. PPO is told the truth: the stored
log-probability is the mixture's, not the network's, so the importance ratio
compares like with like. Salimans & Chen (2018) got Montezuma's Revenge from one
demonstration this way; the update stays outcome-driven, the label only lowers
the cost of *trying* the good move.

**Real code** — `guided_probs` (`src/model/training.py`):

```python
b = p.clone()
for i, demo in enumerate(demos):
    if demo:
        idx = torch.tensor(sorted(demo), dtype=torch.long)
        b[i] = (1.0 - eps) * p[i]
        b[i, idx] += eps / len(idx)
```

In [ ]:
# TASK: guided_mixture
def mixture(p, demo, eps):
    """p: 1-D tensor of network probabilities over actions; demo: set of action ids; eps in [0,1).
    Return the behaviour distribution b (same shape). Rows without a demo are the identity."""
    b = p.clone()
    if demo:
        ...
    return b

# By hand: 20 legal moves, the network gives the mate 0.1, eps 0.5, one demo move (the mate).
# Behaviour probability of the mate?  And of one particular non-mate move that had 0.9/19 before?
MY_MATE_PROB = ...
MY_OTHER_PROB = ...

In [ ]:
from src.model.training import guided_probs
_p = torch.full((1, 20), 0.9 / 19); _p[0, 7] = 0.1
_mine = mixture(_p[0], {7}, 0.5); _ref = guided_probs(_p, [{7}], 0.5)[0]
check("mixture matches guided_probs", close(_mine, _ref, 1e-6))
check("mixture sums to one", close([_mine.sum().item()], [1.0], 1e-6))
check("hand probabilities", close([MY_MATE_PROB, MY_OTHER_PROB], [_ref[7].item(), _ref[0].item()]), f"engine: {_ref[7].item():.4f}, {_ref[0].item():.4f}")

### 2b. Who made that move? Three arms, one bug each

Technique boards (section 4) advance their curriculum only on *clean* wins, wins
the network made without the demonstrator. Three arms tried three ways of
deciding "clean". Plot: the queen-versus-king level over 120 updates in each,
names hidden.

- one arm counted **every** win as clean;
- one flagged an episode as teacher-made whenever the *played move equalled* a demo move, even if the network chose it itself;
- one flips an explicit coin with probability epsilon and flags only when the coin said "demonstrator".

In [ ]:
_arms3 = {"CURSIL": "experiments/endgame-technique/CURSIL/run.json", "CURSIL2": "experiments/endgame-technique/CURSIL2/run.json",
          "CURSIL3": "experiments/endgame-technique/CURSIL3/run.json"}
_rng = random.Random(3); _let = list("KLM"); _rng.shuffle(_let)
HIDDEN3 = dict(zip(_let, _arms3))
plt.figure(figsize=(7, 4))
for letter, arm in HIDDEN3.items():
    xs, ys = series(load_run(_arms3[arm])[0], "technique_level_Q"); plt.plot(xs, ys, marker="o", label=letter)
plt.title("technique_level_Q (queen v king): 0 corner ... 3 random"); plt.xlabel("update"); plt.legend(); plt.show()

In [ ]:
# TASK: read_clean
# Map each letter to the rule it used: "every_win", "flag_by_action" or "coin".
MY_RULES = {"K": ..., "L": ..., "M": ...}

In [ ]:
_truth3 = {k: {"CURSIL": "every_win", "CURSIL2": "flag_by_action", "CURSIL3": "coin"}[v] for k, v in HIDDEN3.items()}
check("clean-episode rules identified", MY_RULES == _truth3, f"truth {_truth3}")
print("why: counting every win races the ladder to level 3 on teacher-made mates; flagging by action marks every"
      " mating move as the teacher's (the mate IS the demo), so no win is ever clean and the ladder freezes at 0.")

## 3. Beating yesterday's self is not getting better

**Plain words.** In self-play the opponent is always the current network. Balduzzi
et al. (2019): that produces a sequence of stronger agents only when the game is
roughly transitive; otherwise you get cycles, and "against whom" strength is
measured becomes unclear. Bansal et al. (2018) probe this with a win matrix:
every checkpoint plays every other. The first matrix here (2026-09-17) put the
supervised prior against five RL checkpoints that had each "improved" on their
predecessor.

**Real code** — `score_from_counts` (`src/eval/matches.py`): counts are keyed
`a_<colour>_<result>` where colour is the colour model A played.

```python
RESULT_SCORE_WHITE = {"white_win": 1.0, "black_win": 0.0, "draw": 0.5, "unfinished": 0.5}
for key, n in counts.items():
    colour, result = key.split("_", 2)[1], key.split("_", 2)[2]
    s = RESULT_SCORE_WHITE[result]
    points += n * (s if colour == "white" else 1 - s)
    total += n
return points / total
```

In [ ]:
# TASK: match_score
def match_score(counts):
    """Score of model A in [0, 1] from a counts dict like
    {"a_white_white_win": 3, "a_white_draw": 5, "a_black_black_win": 2, "a_black_white_win": 6, ...}."""
    points = total = 0
    for key, n in counts.items():
        colour, result = key.split("_", 2)[1], key.split("_", 2)[2]
        ...
    return points / total

# By hand: A as white: 3 wins, 5 draws, 2 losses; A as black: 2 wins (black_win), 6 losses (white_win), 2 draws.
MY_SCORE = ...

In [ ]:
from src.eval.matches import score_from_counts
_c = {"a_white_white_win": 3, "a_white_draw": 5, "a_white_black_win": 2, "a_black_black_win": 2, "a_black_white_win": 6, "a_black_draw": 2}
check("match_score matches the engine", close([match_score(_c)], [score_from_counts(_c)], 1e-9), f"engine {score_from_counts(_c):.3f}")
check("hand score", close([MY_SCORE], [score_from_counts(_c)]))

In [ ]:
def show_matrix(path, title):
    d = json.loads(Path(path).read_text()); names = d["names"]; t = d["table"]
    print(title); print(" " * 12 + "".join(f"{n:>12s}" for n in names))
    for a in names:
        print(f"{a:>12s}" + "".join(f"{t[a][b]:12.2f}" if b != a else f"{'-':>12s}" for b in names))
    return d
M1 = show_matrix("experiments/win-matrix/matrix.json", "2026-09-17: SL prior vs five RL checkpoints (row score vs column)")
print()
M2 = show_matrix("experiments/win-matrix/matrix_pool_long.json", "after 800 updates with the opponent pool")

In [ ]:
# TASK: read_matrix
# 1. In the first matrix, which model has the best row (mean score against everyone else)?
BEST_ROW_M1 = ...
# 2. The five RL checkpoints each "beat" their predecessor in training. In the first matrix, how do they do
#    against each other? Fill the mean of all RL-vs-RL scores rounded to 1 decimal (0.5 = nobody is better).
RL_VS_RL_MEAN = ...
# 3. One sentence: what does the pair (answer 1, answer 2) say about what self-play was optimising?
DRIFT_SENTENCE = """..."""

In [ ]:
def _row_mean(d, a): return sum(v for b, v in d["table"][a].items()) / (len(d["names"]) - 1)
_best = max(M1["names"], key=lambda a: _row_mean(M1, a))
_rl = [n for n in M1["names"] if n != "SL"]
_rlmean = sum(M1["table"][a][b] for a in _rl for b in _rl if a != b) / (len(_rl) * (len(_rl) - 1))
check("best row", BEST_ROW_M1 == _best, f"engine: {_best} ({_row_mean(M1, _best):.2f})")
check("RL vs RL mean", close([RL_VS_RL_MEAN], [round(_rlmean, 1)]), f"engine: {_rlmean:.3f}")
print("your sentence:", DRIFT_SENTENCE)
print("expected direction: the checkpoints are at par with each other and all lose to the prior: self-play optimised"
      " beating the current self, and the current self had drifted away from the prior's strength (Balduzzi: non-transitive).")

## 4. Frozen copies of yourself as sparring partners

**Plain words.** Bansal et al. (2018) sample the opponent from the history of
checkpoints rather than the latest one; the latest-only setting was the worst in
every one of their environments. Here the pool holds the supervised prior plus
the last four snapshots of the learner, one taken every 50 updates, and each pool
board draws a member uniformly when its game ends. The learner's colour is drawn
too. Only the learner's own plies enter the PPO batch; the frozen member's plies
are part of the environment, exactly as section 2 of the first workshop argued.

**Real code** — `OpponentPool` (`src/training/opponent_pool.py`):

```python
@property
def members(self):
    out = [("prior", self.prior)] if self.prior is not None else []
    return out + list(self._snaps)            # deque(maxlen=size)

def maybe_snapshot(self, model, update):
    if self.size <= 0 or self.snapshot_every <= 0 or update % self.snapshot_every != 0:
        return False
    self._snaps.append((f"snap_{update}", freeze(model)))
    return True

def sample(self):
    return self._rng.choice(self.members)
```

In [ ]:
# TASK: pool_members
# Pool: prior present, size 4, snapshot_every 50. After update 120 and after update 300:
# how many members are there, and what is the probability that a pool board draws the prior?
MEMBERS_AT_120 = ...
PRIOR_PROB_AT_120 = ...
MEMBERS_AT_300 = ...
PRIOR_PROB_AT_300 = ...
# Which snapshot names are in the pool after update 300? (a sorted list of strings like "snap_50")
SNAPS_AT_300 = [...]

In [ ]:
from src.training.opponent_pool import OpponentPool
_pool = OpponentPool(torch.nn.Linear(1, 1), size=4, snapshot_every=50, seed=0)
_m = {}
for u in range(1, 301):
    _pool.maybe_snapshot(torch.nn.Linear(1, 1), u)
    if u in (120, 300): _m[u] = [n for n, _ in _pool.members]
check("members at 120", (MEMBERS_AT_120, PRIOR_PROB_AT_120) == (len(_m[120]), 1 / len(_m[120])), f"engine: {_m[120]}")
check("members at 300", (MEMBERS_AT_300, PRIOR_PROB_AT_300) == (len(_m[300]), 1 / len(_m[300])), f"engine: {_m[300]}")
check("snapshot names at 300", sorted(SNAPS_AT_300) == sorted(n for n in _m[300] if n != "prior"))

In [ ]:
_pl = load_run("experiments/opponent-pool/POOL_LONG/run.json")[0]
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
xs, ys = series(_pl, "prior_score"); axes[0].plot(xs, ys, marker="o"); axes[0].axhline(0.5, ls="--", c="grey"); axes[0].set_title("POOL_LONG: score vs the prior (20 games, eval)")
for k in ("pool_score_prior",): 
    xs, ys = series(_pl, k); axes[1].plot(xs, ys, label=k)
axes[1].set_title("training boards: learner score vs the prior member"); axes[1].legend()
for ax in axes: ax.set_xlabel("update")
plt.show()

In [ ]:
# TASK: read_pool
# "Par with the prior" was defined as prior_score 0.5. Did POOL_LONG reach it at any evaluation?  True / False
REACHED_PAR = ...
# The second matrix above (M2) has three RL checkpoints. Are they now ordered (a later one clearly beats an
# earlier one, > 0.55), or still at par with each other? "ordered" / "par"
RL_ORDER_AFTER_POOL = ...

In [ ]:
_reached = any(y >= 0.5 for y in series(_pl, "prior_score")[1])
check("reached par", REACHED_PAR is _reached, f"max prior_score {max(series(_pl, 'prior_score')[1]):.3f}")
_rl2 = [n for n in M2["names"] if n != "SL"]
_ordered = any(M2["table"][a][b] > 0.55 for a in _rl2 for b in _rl2 if a != b)
check("RL order after pool", RL_ORDER_AFTER_POOL == ("ordered" if _ordered else "par"), f"RL-vs-RL scores: {[(a, b, round(M2['table'][a][b], 2)) for a in _rl2 for b in _rl2 if a != b]}")

## 5. Bare-king endgames and a curriculum without a ruler

**Plain words.** Lichess has 916,763 endgame puzzles and not one "queen versus
bare king" position: humans resign those. So the engine generates them: queen,
rook, two rooks, queen-and-rook against a lone king. A held-out network alone
finished queen-versus-king 0 times out of 50. Florensa et al. widen the start
distribution from the goal outwards using a *good starts* band: keep starts whose
success rate is between `R_min` (0.1, not hopeless) and `R_max` (0.9, not
solved), replay a few solved ones so they are not forgotten, and probe a few
hard ones. Their ruler is a reverse dynamics model; chess has none, so the
stand-in is geometry: 36 buckets by the weak king's distance to the edge, the
kings' distance, and the pieces' distance.

**Real code** — `GoodStartsCurriculum` (`src/envs/technique.py`):

```python
def status(self, label, bucket):
    rt = self.rate(label, bucket)                  # None until `window` reports
    if rt is None:      return "unknown"
    if rt > self.r_max: return "graduated"
    if rt < self.r_min: return "hard"
    return "good"

def weights(self, label):
    ... out.append({"unknown": unknown_w, "good": 1.0,
                    "graduated": self.replay_share, "hard": self.probe_share}[st])
```

In [ ]:
# TASK: good_starts
def bucket_weight(rate, r_min, r_max, replay_share, probe_share, unknown_w=1.0):
    """Sampling weight of one bucket from its windowed success rate (None = not enough data)."""
    if rate is None:
        return unknown_w
    ...

# By hand, r_min 0.1, r_max 0.9, replay 0.2, probe 0.1, warm start off: rates None, 0.95, 0.5, 0.05 ->
MY_WEIGHTS = [..., ..., ..., ...]

In [ ]:
from src.envs.technique import GoodStartsCurriculum
_g = GoodStartsCurriculum(["Q"], r_min=0.1, r_max=0.9, window=20, replay_share=0.2, probe_share=0.1, warm_start=False)
for bucket, rate in ((1, 0.95), (2, 0.5), (3, 0.05)):
    for i in range(20): _g.report("Q", bucket, i < round(rate * 20))
_ref_w = _g.weights("Q")[:4]
_mine_w = [bucket_weight(r, 0.1, 0.9, 0.2, 0.1) for r in (None, 0.95, 0.5, 0.05)]
check("bucket_weight matches the engine", close(_mine_w, _ref_w), f"engine {_ref_w}")
check("hand weights", close(MY_WEIGHTS, _ref_w))

### 5b. Why the technique demonstrator is not cheating

The demonstrator on technique boards is the rules engine: a mate in one, or a
forcing mate in two. It is *external knowledge*, but only one or two plies of it.
The thirty plies of approach, driving the king to the edge, come from nowhere
but the outcome. That is why the level ladder mattered: at level 0 the king is
already cornered and the approach is two moves long.

In [ ]:
_l3 = load_run("experiments/endgame-technique/LONG3/run.json")[0]
print("LONG3 held-out technique rate at the hardest level:", {k: round(_l3[-1][k], 3) for k in ("technique_rate_Q", "technique_rate_R", "technique_rate_RR", "technique_rate_QR")})
print("LONG3 training, queen v king: all wins", round(_l3[-1]["technique_success_rate_Q"], 2), "clean wins", round(_l3[-1]["technique_clean_success_rate_Q"], 2))

In [ ]:
# TASK: read_technique
# One sentence: the training success rate for Q (technique_success_rate_Q) is higher than the *clean* one
# (technique_clean_success_rate_Q). Which of the two should the curriculum use, and why?
CLEAN_OR_ALL = ...           # "clean" or "all"
WHY_CLEAN = """..."""

In [ ]:
check("curriculum uses", CLEAN_OR_ALL == "clean")
print("your sentence:", WHY_CLEAN)
print(f"LONG3 training: all {_l3[-1]['technique_success_rate_Q']:.2f} vs clean {_l3[-1]['technique_clean_success_rate_Q']:.2f};"
      " the gap is the demonstrator's contribution, and a ladder that climbs on it climbs to positions the network cannot solve.")

## 6. Learn from your own rare wins more than once

**Plain words.** A won technique game is seen once by PPO and then thrown away,
and the thirty approach plies get a return of `gamma^30` of the win. Oh et al.
(2018) keep every learner ply with its eventual Monte Carlo return `R` in a
replay buffer and, after each PPO update, draw minibatches with priority
`(R - V)+` and train on the *self-imitation* loss: policy `-log pi(a|s) (R - V)+`,
value `1/2 ((R - V)+)^2`. Only plies whose past return beat the current value
estimate get gradient. No importance ratio: the objective is a lower bound valid
for any past behaviour that did better than the learner. Once the value head
catches up, `(R - V)+` is zero and the loss switches itself off (record 0015).

**Real code** — `sil_loss` (`src/training/sil.py`):

```python
adv = (ret - value.detach()).clamp(min=0)
logp = Categorical(probs=probs).log_prob(action)
policy = (-logp * adv * weights).mean()
value_term = 0.5 * (((ret - value).clamp(min=0)) ** 2 * weights).mean()
```

In [ ]:
# TASK: sil_terms
def sil_terms(logp, ret, value):
    """Per-sample (policy_term, value_term) of the SIL loss, weights 1:
    policy = -logp * (ret - value)+ ; value = 0.5 * ((ret - value)+)^2."""
    adv = ...
    return ..., ...

# By hand, log pi(a|s) = log 0.2:
#   (a) R = 2, V = 0.5      (b) R = -2, V = 0.1      (c) R = 2, V = 2.5
MY_POLICY_TERMS = [..., ..., ...]     # -log(0.2) * (R-V)+  for a, b, c

In [ ]:
_lp = math.log(0.2)
_ref = [(-_lp * max(r - v, 0), 0.5 * max(r - v, 0) ** 2) for r, v in ((2, 0.5), (-2, 0.1), (2, 2.5))]
_mine = [sil_terms(_lp, r, v) for r, v in ((2, 0.5), (-2, 0.1), (2, 2.5))]
check("sil_terms", close([x for t in _mine for x in t], [x for t in _ref for x in t]), f"engine {[(round(a, 4), round(b, 4)) for a, b in _ref]}")
check("hand policy terms", close(MY_POLICY_TERMS, [t[0] for t in _ref]))
print("only (a) trains: (b) lost and (c) already valued higher than the return it got.")

### 6b. Returns for the buffer: gamma once per own ply

The accumulator holds a board's plies until the game ends, then writes each ply's
return as `terminal * gamma^k`, where `k` counts the mover's *own* remaining plies
including this one. So the ply that mates gets `gamma * terminal`, same convention
as the value targets.

In [ ]:
# TASK: sil_returns
def buffer_returns(sides, tr_white, tr_black, gamma):
    """sides: list of players per stored ply in order (1 = white, 0 = black).
    Return the list of returns as EpisodeAccumulator._finish computes them."""
    remaining = {1: 0, 0: 0}
    out = [0.0] * len(sides)
    for k in range(len(sides) - 1, -1, -1):
        ...
    return out

# By hand: plies W B W B W, white mates on the last ply (terminal white +2, black -2), gamma 0.99.
MY_RETURNS = [..., ..., ..., ..., ...]

In [ ]:
from src.training.sil import ReplayBuffer, EpisodeAccumulator
_buf = ReplayBuffer(16); _acc = EpisodeAccumulator(1, 0.99, _buf)
_sides = [1, 0, 1, 0, 1]
_acc.pending[0] = [(torch.zeros(20, 8, 8), torch.ones(4674), 0, s, 0.0) for s in _sides]
_acc._finish(0, 2.0, -2.0)
_ref = [float(x) for x in _buf.ret[:5]]
check("buffer_returns matches the accumulator", close(buffer_returns(_sides, 2.0, -2.0, 0.99), _ref), f"engine {[round(x, 4) for x in _ref]}")
check("hand returns", close(MY_RETURNS, _ref))

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for k in ("sil_positive_share", "sil_valid_share"):
    xs, ys = series(_fin, k); ax.plot(xs, ys, label=k)
ax.set_title("LONG2+LONG3: share of buffer plies with (R - V)+ > 0, and share among drawn"); ax.legend(); ax.set_xlabel("update"); plt.show()

In [ ]:
# TASK: read_sil
# sil_positive_share is the share of the whole buffer with positive priority; sil_valid_share is the share
# among the plies actually drawn. One is near 1.0 all run, the other is much lower. Which is which, and
# why is that not a contradiction? (Hint: how are minibatches drawn?)
NEAR_ONE = ...          # "sil_positive_share" or "sil_valid_share"
WHY_NOT_CONTRADICTION = """..."""

In [ ]:
_pos = series(_fin, "sil_positive_share")[1]; _val = series(_fin, "sil_valid_share")[1]
_near = "sil_valid_share" if sum(_val) / len(_val) > sum(_pos) / len(_pos) else "sil_positive_share"
check("near one", NEAR_ONE == _near, f"means: positive {sum(_pos)/len(_pos):.2f}, valid {sum(_val)/len(_val):.2f}")
print("your sentence:", WHY_NOT_CONTRADICTION)
print("expected: minibatches are drawn with probability proportional to (R - V)+, so almost every drawn ply is valid"
      " even when most of the buffer has priority zero; the buffer share is what tells you how much is left to learn.")

## 7. Lambda 1 when the only reward is at the end

**Plain words.** Section 5 of the first workshop said lambda 0.95 "leans on the
observed future for a dozen moves, then trusts the value head". On technique
boards the reward is only at the mate, thirty plies away, and the value head is
the weakest part of the network there. With lambda 0.95 a ply ten own moves
before the mate gets `(gamma * lambda)^10` of the outcome's credit; with lambda 1
it gets `gamma^10` and the rest is not replaced by a value guess. The ablation
(arm LAMBDA vs GSL2) gave more decisive games and more in-game mates, and
lambda 1 became the default on 2026-09-18.

In [ ]:
# TASK: lambda_credit
def credit(gamma, lam, k):
    """Weight of the terminal outcome in the advantage of a ply k own moves before the end (GAE, terminal-only reward)."""
    return ...

MY_CREDIT_095 = ...     # gamma 0.99, lambda 0.95, k 10
MY_CREDIT_1 = ...       # gamma 0.99, lambda 1.0, k 10

In [ ]:
check("credit formula", close([credit(0.99, 0.95, 10), credit(0.99, 1.0, 10)], [(0.99 * 0.95) ** 10, 0.99 ** 10]))
check("hand credits", close([MY_CREDIT_095, MY_CREDIT_1], [(0.99 * 0.95) ** 10, 0.99 ** 10]), f"{(0.99*0.95)**10:.3f} vs {0.99**10:.3f}")

## 8. Does the opponent punish gifts?

**Plain words.** LONG3 beat LONG2 19 to 10 and hung a piece every ten plies. The
owner's hypothesis: the self-play defender does not punish stupid moves, so the
learner never learns to avoid them. The audit asks, for every one of our moves,
two one-ply rules questions: did it hand the opponent a mate in one it could have
avoided, or a piece worth at least a minor piece after the cheapest recapture?
Then: did the opponent take it? AlphaStar's league answers this class of problem
with *exploiter* agents whose only job is to expose the main agent's weaknesses;
a rules referee that always takes the free piece is a hand-made exploiter.

**Real code** — `capture_gain` (`src/envs/open_spiel_env.py`):

```python
victim = board.piece_at(move.to_square)
gained = PIECE_VALUE[victim.piece_type] if victim else 1        # en passant
board.push(move)
recapturable = any(m.to_square == move.to_square for m in board.legal_moves)
board.pop()
return gained - (PIECE_VALUE[board.piece_at(move.from_square).piece_type] if recapturable else 0)
```

In [ ]:
# TASK: net_gain
PIECE_VALUE = {"P": 1, "N": 3, "B": 3, "R": 5, "Q": 9}
def net_gain(victim, attacker, recapturable):
    """One-exchange net material for a capture: victim value minus attacker value if the square can be recaptured."""
    return ...

# By hand: (a) knight takes an undefended queen; (b) bishop takes a knight defended by a rook;
# (c) rook takes a defended pawn. Which are "gifts" at threshold 3?
MY_GAINS = [..., ..., ...]
MY_GIFTS = [..., ..., ...]     # True / False each

In [ ]:
import chess
from src.envs.open_spiel_env import capture_gain
_cases = [("4k3/8/2n5/8/3Q4/8/8/4K3 b - - 0 1", "Nxd4"),        # knight takes an undefended queen
          ("4k3/2r5/2n5/8/4B3/8/8/4K3 w - - 0 1", "Bxc6"),       # bishop takes a knight the rook defends
          ("4k3/8/8/3b4/4p3/8/8/4R2K w - - 0 1", "Rxe4")]        # rook takes a pawn the bishop defends
_ref = []
for fen, san in _cases:
    b = chess.Board(fen); _ref.append(capture_gain(b, b.parse_san(san)))
_mine = [net_gain("Q", "N", False), net_gain("N", "B", True), net_gain("P", "R", True)]
check("net_gain matches capture_gain", _mine == _ref, f"engine {_ref}")
check("hand gains", MY_GAINS == _ref)
check("hand gifts", MY_GIFTS == [g >= 3 for g in _ref])

In [ ]:
_audit = json.loads(Path("experiments/blunder-audit/long3.json").read_text())["defenders"]
print(f"{'defender':12s} {'mate gifts/100':>15s} {'punished':>9s} {'piece gifts/100':>16s} {'punished':>9s}")
for n, r in _audit.items():
    print(f"{n:12s} {r['gift_mate_per100']:15.2f} {r['punish_rate_mate']:9.2f} {r['gift_material_per100']:16.2f} {r['punish_rate_material']:9.2f}")

In [ ]:
# TASK: read_audit
# 1. If the network punishes a hung piece with base rate b on its own, and the punisher fires with probability p
#    whenever a punishing move exists (and always punishes when it fires), what is the punish rate?
def punish_rate(b, p):
    return ...
MY_RATE_MATERIAL = ...     # b = 0.54 (self row), p = 0.5
MY_RATE_MATE = ...         # b = 0.13, p = 0.5
# 2. Why does swapping the self-play defender for the SL prior NOT fix this? One sentence, from the table.
WHY_NOT_PRIOR = """..."""

In [ ]:
check("punish_rate formula", close([punish_rate(0.5, 0.5), punish_rate(0.0, 1.0), punish_rate(0.2, 0.0)], [0.75, 1.0, 0.2]))
_b_mat, _b_mate = _audit["self"]["punish_rate_material"], _audit["self"]["punish_rate_mate"]
check("hand rates", close([MY_RATE_MATERIAL, MY_RATE_MATE], [_b_mat + 0.5 * (1 - _b_mat), _b_mate + 0.5 * (1 - _b_mate)], 0.02), f"{_b_mat + 0.5*(1-_b_mat):.3f}, {_b_mate + 0.5*(1-_b_mate):.3f}")
print("your sentence:", WHY_NOT_PRIOR)
print(f"expected: the prior punishes free pieces at {_audit['SL']['punish_rate_material']:.2f} and mates at {_audit['SL']['punish_rate_mate']:.2f}, no better than we do.")

## 9. Calibrating the finishing dial: where exactly does it break?

**Plain words.** A dial that reads 0.12 tells you little until you know what a
good answer would read. The calibration plays finishing starts three ways: our
network sampling, our network greedy, and a *human-line attacker* that plays the
human's recorded moves while the defender stays on the record and our greedy move
once it leaves. The third is a ceiling for "right first move, then us".

In [ ]:
_cal = json.loads(Path("experiments/finishing-calibration/long3.json").read_text())["depths"]
print(f"{'depth':>6s} {'sampled':>9s} {'greedy':>8s} {'human-line':>11s} {'defender stayed on line':>25s}")
for d, r in _cal.items():
    print(f"{d:>6s} {r['sampled']['rate']:9.2f} {r['greedy']['rate']:8.2f} {r['human']['rate']:11.2f} {r['human']['defender_stayed_on_line']:25.2f}")

In [ ]:
# TASK: read_calibration
# 1. At which depth does the human-line attacker still roughly double our conversion rate? (2, 10 or 20)
HUMAN_HELPS_AT = ...
# 2. At depth 20 the human line does not help. Look at "defender stayed on line": how many plies of the
#    human plan does the attacker actually get before it is on its own? One sentence on what the ceiling then means.
CEILING_SENTENCE = """..."""

In [ ]:
_ratio = {d: r["human"]["rate"] / max(r["sampled"]["rate"], 1e-9) for d, r in _cal.items()}
_helps = [int(d) for d, v in _ratio.items() if v >= 1.8]
check("human line helps at", HUMAN_HELPS_AT in _helps and HUMAN_HELPS_AT != 20, f"ratios {dict((d, round(v, 2)) for d, v in _ratio.items())}")
print("your sentence:", CEILING_SENTENCE)
print("expected: the defender leaves the record after about one ply, so the ceiling measures 'human first move, then us';"
      " at depth 20 one good first move is not a plan, and the weakness is plan initiation, not the last moves.")

## 10. Your words

Three to five sentences: what surprised you, which of the new dials (finish
depth, clean technique rate, prior score, punish rate) you would watch first in
the next run, and where outside chess these shapes appear: curricula from
demonstrations in robotics, replaying rare successes in recommender or trading
agents, exploiters and red teams in any system trained against a simulator of its
counterparty. This text goes into `learning/records/`.

In [ ]:
MY_REFLECTION = """
...
"""
print(MY_REFLECTION)
print(f"checks passed: {sum(ok for _, ok in RESULTS)}/{len(RESULTS)}")